# HazardWatch — fire & smoke camera model (D-Fire → YOLO11n)

Trains the model that fills `YOLO_Flame_Detected` / `YOLO_Smoke_Detected`.

* **Runtime:** Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**.
* **Data:** D-Fire (≈21,000 images, YOLO labels). **Class 0 = smoke, 1 = fire** (checked on the labels).
* **Saved to your Google Drive:** `MyDrive/<your HazardWatch folder>/dfire/` (checkpoints every epoch, so a disconnect can resume).
* **Time:** about 2–3 hours for 40 epochs on a free T4. Keep this tab open.
* Ultralytics YOLO is AGPL-3.0 licensed; D-Fire: cite Venâncio et al., Neural Computing and Applications (2022).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."


In [ ]:
# 1. Google Drive (for the D-Fire data, checkpoints and results). Approve the pop-up.
from google.colab import drive
drive.mount('/content/drive')
import os
MY = '/content/drive/MyDrive'
# Use the existing HazardWatch folder whatever its capitalisation (e.g. "Hazardwatch").
HW = next((f'{MY}/{d}' for d in sorted(os.listdir(MY)) if d.lower() == 'hazardwatch'), f'{MY}/HazardWatch')
OUT = f'{HW}/dfire'
os.makedirs(OUT, exist_ok=True)
print('HazardWatch folder:', HW); print('Results will be saved in', OUT)

In [ ]:
!pip -q install ultralytics

In [ ]:
# 2. Get D-Fire from your Drive: an unzipped "D-Fire" folder or a "D-Fire.zip" inside the HazardWatch folder.
import glob, os, shutil, zipfile
from concurrent.futures import ThreadPoolExecutor
DATA = '/content/dfire'
src_dir = next((f'{HW}/{d}' for d in os.listdir(HW) if d.lower() in ('d-fire', 'dfire') and os.path.isdir(f'{HW}/{d}')), None)
src_zip = next((f'{HW}/{d}' for d in os.listdir(HW) if d.lower() in ('d-fire.zip', 'dfire.zip')), None)
assert src_dir or src_zip, f'Put the D-Fire folder (or D-Fire.zip) inside {HW} and run this cell again.'

if src_dir:
    # Many small files: copy them to the fast local disk in parallel (reading from Drive during training is slow).
    files = [p for p in glob.glob(f'{src_dir}/**/*', recursive=True) if os.path.isfile(p)]
    print(len(files), 'files in', src_dir, '- copying to local disk ...')
    def cp(p):
        dst = os.path.join(DATA, os.path.relpath(p, src_dir)); os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst): shutil.copyfile(p, dst)
    with ThreadPoolExecutor(32) as ex: list(ex.map(cp, files))
else:
    print('Unzipping', src_zip, '...'); shutil.copy(src_zip, '/content/D-Fire.zip')
    zipfile.ZipFile('/content/D-Fire.zip').extractall(DATA)

# Find the train / test image folders whatever the layout is, and check the upload is complete.
img_dirs = sorted({os.path.dirname(p) for p in glob.glob(f'{DATA}/**/images/*.jpg', recursive=True)})
print(img_dirs)
train = next(d for d in img_dirs if 'train' in d.lower())
val = next(d for d in img_dirs if 'test' in d.lower() or 'val' in d.lower())
for d in (train, val):
    imgs = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(d + '/*.jpg')}
    labs = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(d.replace('images', 'labels') + '/*.txt')}
    print(f'{d}: {len(imgs)} images, {len(labs)} labels, {len(imgs - labs)} images without a label file')
    assert len(imgs - labs) < 0.01 * len(imgs), 'Many labels are missing: the Drive upload may not be finished.'
# The published D-Fire split has about 17,200 train and 4,300 test images.
print('train/test images:', len(glob.glob(train + '/*.jpg')), len(glob.glob(val + '/*.jpg')))

In [ ]:
# 3. Check the class ids on the labels (D-Fire: 0 = smoke, 1 = fire).
import collections
counts = collections.Counter()
for f in glob.glob(train.replace('images', 'labels') + '/*.txt'):
    for line in open(f):
        if line.strip(): counts[line.split()[0]] += 1
print('boxes per class id:', dict(counts))
assert set(counts) <= {'0', '1'}, 'unexpected class ids'
yaml = f'''path: /
train: {train}
val: {val}
names:
  0: smoke
  1: fire
'''
open('/content/dfire.yaml', 'w').write(yaml); print(yaml)

In [ ]:
# 4. Train YOLO11n (nano: fast enough for a CPU or phone). Resumes automatically after a disconnect.
from ultralytics import YOLO
last = f'{OUT}/train/weights/last.pt'
if os.path.exists(last):
    print('Resuming from', last)
    model = YOLO(last); model.train(resume=True)
else:
    model = YOLO('yolo11n.pt')
    model.train(data='/content/dfire.yaml', epochs=40, imgsz=640, batch=32, workers=2,
                project=OUT, name='train', exist_ok=True, patience=10, save_period=1, seed=0)

In [ ]:
# 5. Evaluate on the D-Fire test set.
best = f'{OUT}/train/weights/best.pt'
m = YOLO(best)
r = m.val(data='/content/dfire.yaml', imgsz=640, batch=32)
import json
summary = {'mAP50': float(r.box.map50), 'mAP50_95': float(r.box.map),
           'per_class_mAP50': {m.names[i]: float(v) for i, v in enumerate(r.box.ap50)},
           'precision': float(r.box.mp), 'recall': float(r.box.mr)}
print(json.dumps(summary, indent=2)); open(f'{OUT}/results.json', 'w').write(json.dumps(summary, indent=2))

In [ ]:
# 6. Export for the PC (ONNX, CPU) and keep everything in Drive.
onnx = m.export(format='onnx', imgsz=640, simplify=True)
!cp {onnx} {OUT}/
print('Done. Files in Google Drive:', OUT); print(os.listdir(OUT))